# Week 6: Geometric Transforms — Lab Practice

**Topics:** Transformation Matrices, Affine Transforms, Homography & Document Scanning

This notebook accompanies the Week 6 lecture slides. We focus on two core skills:
1. **Building and applying geometric transformation matrices** — rotation, scaling, translation in homogeneous coordinates
2. **Homography** — computing a perspective transform from 4 point correspondences to "scan" a document

We also include a brief demo of the **Hough Transform** for line detection.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

%matplotlib inline

### Environment setup

This notebook works both **locally** and on **Google Colab**.

In [ ]:
import os, urllib.request

# Detect Google Colab
IN_COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, "__IPYTHON__") else False

if IN_COLAB:
    REPO_URL = "https://raw.githubusercontent.com/HyeongminLEE/image-processing-tutorial/main"
    os.makedirs("images", exist_ok=True)
    for fname in ["parrots_square.jpg", "yangbanga.jpg", "calligraphy_hall.jpg"]:
        url = f"{REPO_URL}/images/{fname}"
        if not os.path.exists(f"images/{fname}"):
            urllib.request.urlretrieve(url, f"images/{fname}")
            print(f"Downloaded {fname}")
    IMG_DIR = "images/"
else:
    IMG_DIR = "../images/"

print(f"Running on: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Image directory: {IMG_DIR}")

### Display helpers

Utility functions used throughout this notebook. Each helper displays **one item** per call.

In [ ]:
def show_image(img, title=None, scale=4):
    """Display a single image.

    Grayscale (2-D) arrays automatically use a gray colormap with [0, 255].
    """
    fig, ax = plt.subplots(figsize=(scale, scale))
    if img.ndim == 2:
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    else:
        ax.imshow(img)
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_transform(pts_before, pts_after, title=None, scale=5):
    """Show original and transformed 2D points on the same axes.

    Points are connected as a closed polygon.
    Blue = original, Red = transformed.
    """
    fig, ax = plt.subplots(figsize=(scale, scale))
    for pts, color, marker, label in [
        (pts_before, "royalblue", "o", "Original"),
        (pts_after, "crimson", "s", "Transformed"),
    ]:
        closed = np.vstack([pts, pts[0:1]])
        ax.plot(closed[:, 0], closed[:, 1], f"-{marker}",
                color=color, markersize=8, label=label)
    all_pts = np.vstack([pts_before, pts_after])
    margin = 1.0
    ax.set_xlim(all_pts[:, 0].min() - margin, all_pts[:, 0].max() + margin)
    ax.set_ylim(all_pts[:, 1].min() - margin, all_pts[:, 1].max() + margin)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color="k", linewidth=0.5)
    ax.axvline(0, color="k", linewidth=0.5)
    ax.legend()
    if title:
        ax.set_title(title)
    plt.tight_layout()
    plt.show()

---
## 0. Hough Transform — Quick Demo

The **Hough Transform** detects parametric shapes (lines, circles) in edge maps using a **voting** scheme. Each edge pixel votes for all parameter combinations that could pass through it; peaks in the accumulator reveal the dominant shapes.

This is a brief demo — no exercises. See the lecture slides for the full theory (Cartesian vs polar parameterization, accumulator, etc.).

In [ ]:
# Load the same image used in the lecture slides (calligraphy hall sign)
img_hough = np.array(Image.open(IMG_DIR + "calligraphy_hall.jpg"))
gray_hough = np.array(Image.open(IMG_DIR + "calligraphy_hall.jpg").convert("L"))

edges = cv2.Canny(gray_hough, 50, 150)

show_image(gray_hough, title="Grayscale")
show_image(edges, title="Canny edges")

In [ ]:
# Standard Hough Transform (same code as lecture slides)
lines = cv2.HoughLines(edges, rho=1, theta=np.pi / 180, threshold=150)

# Convert (rho, theta) to endpoints and draw
result = img_hough.copy()
for line in lines:
    rho, theta = line[0]
    a, b = np.cos(theta), np.sin(theta)
    x0, y0 = a * rho, b * rho
    pt1 = (int(x0 + 1000 * (-b)), int(y0 + 1000 * (a)))
    pt2 = (int(x0 - 1000 * (-b)), int(y0 - 1000 * (a)))
    cv2.line(result, pt1, pt2, (255, 0, 0), 2)

print(f"Lines detected: {len(lines)}")
show_image(result, title=f"Hough lines ({len(lines)} detected)", scale=5)

The `threshold` parameter controls how many votes a line needs to be detected. Too low → many false detections; too high → misses real lines.

In [ ]:
# Compare different threshold values
def draw_hough_lines(img, edges, threshold):
    """Run HoughLines and draw results. Returns image and line count."""
    lines = cv2.HoughLines(edges, rho=1, theta=np.pi / 180, threshold=threshold)
    vis = img.copy()
    n = 0
    if lines is not None:
        n = len(lines)
        for line in lines:
            rho, theta = line[0]
            a, b = np.cos(theta), np.sin(theta)
            x0, y0 = a * rho, b * rho
            pt1 = (int(x0 + 1000 * (-b)), int(y0 + 1000 * (a)))
            pt2 = (int(x0 - 1000 * (-b)), int(y0 - 1000 * (a)))
            cv2.line(vis, pt1, pt2, (255, 0, 0), 2)
    return vis, n

vis_80, n_80 = draw_hough_lines(img_hough, edges, threshold=80)
vis_150, n_150 = draw_hough_lines(img_hough, edges, threshold=150)
vis_200, n_200 = draw_hough_lines(img_hough, edges, threshold=200)

show_image(vis_80, title=f"threshold=80  ({n_80} lines)", scale=4)
show_image(vis_150, title=f"threshold=150 ({n_150} lines)", scale=4)
show_image(vis_200, title=f"threshold=200 ({n_200} lines)", scale=4)

---
## 1. Geometric Transforms

A geometric transform maps every point $(x, y)$ to a new location $(x', y')$. We express this mapping as **matrix multiplication**:

$$\begin{pmatrix}x'\\y'\end{pmatrix} = \begin{pmatrix}a & b\\c & d\end{pmatrix}\begin{pmatrix}x\\y\end{pmatrix}$$

Different values of the matrix entries produce different transforms: rotation, scaling, shear, and more. We will build these matrices by hand and apply them to both **points** and **images**.

### Transforms on points

Let's start with a simple triangle and see how different matrices transform it.

In [ ]:
# Define a triangle as 3 points (each row is [x, y])
triangle = np.array([
    [1, 0],
    [0, 2],
    [-1, 0],
], dtype=np.float64)

print(f"Triangle vertices:\n{triangle}")

### Rotation

The rotation matrix for angle $\theta$:

$$R = \begin{pmatrix}\cos\theta & -\sin\theta\\\sin\theta & \cos\theta\end{pmatrix}$$

To transform all points at once: each point is a column, so we compute $R \cdot P^T$ and transpose back.

In [ ]:
theta = np.radians(45)

R = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

print(f"Rotation matrix (theta = 45 deg):\n{R}")

rotated = (R @ triangle.T).T

show_transform(triangle, rotated, title="Rotation by 45 deg")

### Scaling

A diagonal matrix scales $x$ and $y$ independently:

$$S = \begin{pmatrix}s_x & 0\\0 & s_y\end{pmatrix}$$

In [ ]:
S = np.array([
    [2.0, 0],
    [0,   0.5]
])

print(f"Scaling matrix:\n{S}")

scaled = (S @ triangle.T).T

show_transform(triangle, scaled, title="Scale: x * 2, y * 0.5")

### The problem: translation

Rotation and scaling are **matrix multiplication**. But translation is **addition**:

$$\begin{pmatrix}x'\\y'\end{pmatrix} = \begin{pmatrix}x + t_x\\y + t_y\end{pmatrix}$$

This breaks chaining — we cannot combine rotation and translation into a single $2 \times 2$ matrix. The solution: **homogeneous coordinates**.

### Homogeneous coordinates

Add a third coordinate (always 1) to every point:

$$\begin{pmatrix}x\\y\end{pmatrix} \longrightarrow \begin{pmatrix}x\\y\\1\end{pmatrix}$$

Now **all transforms** — including translation — become $3 \times 3$ matrix multiplication:

$$\begin{pmatrix}x'\\y'\\1\end{pmatrix} = \begin{pmatrix}1&0&t_x\\0&1&t_y\\0&0&1\end{pmatrix}\begin{pmatrix}x\\y\\1\end{pmatrix} = \begin{pmatrix}x + t_x\\y + t_y\\1\end{pmatrix}$$

In [ ]:
# Convert to homogeneous coordinates: [x, y] → [x, y, 1]
triangle_h = np.hstack([triangle, np.ones((3, 1))])

print(f"Homogeneous coordinates:\n{triangle_h}")

In [ ]:
# Translation matrix (3x3)
tx, ty = 3, 2

T = np.array([
    [1, 0, tx],
    [0, 1, ty],
    [0, 0, 1],
], dtype=np.float64)

print(f"Translation matrix (tx={tx}, ty={ty}):\n{T}")

translated_h = (T @ triangle_h.T).T
print(f"\nHomogeneous result:\n{translated_h}")

# Convert back to 2D: divide by w (3rd component), then take first 2
w_col = translated_h[:, 2:3]       # w = 1 for affine, but good practice
translated = translated_h[:, :2] / w_col

show_transform(triangle, translated, title=f"Translation by ({tx}, {ty})")

### Combining rotation and translation

With $3 \times 3$ homogeneous matrices, we can chain any transforms by multiplying them into a single matrix: $M = T \cdot R$.

In [ ]:
# Rotate 45 deg, then translate by (3, 2)
theta = np.radians(45)
R_h = np.array([
    [np.cos(theta), -np.sin(theta), 0],
    [np.sin(theta),  np.cos(theta), 0],
    [0,              0,             1]
])

# Reuse T from above (tx=3, ty=2)
M_combined = T @ R_h    # rotate first, then translate

result_h = (M_combined @ triangle_h.T).T
result = result_h[:, :2] / result_h[:, 2:3]

print(f"Combined matrix (T @ R):\n{np.round(M_combined, 3)}")

show_transform(triangle, result, title="Rotate 45 deg + Translate (3, 2)")

### Transforms on images

To warp an entire image, we use `cv2.warpAffine(img, M, (width, height))`.

- `M` is the **$2 \times 3$** affine matrix (the last row $[0, 0, 1]$ is implicit)
- OpenCV uses **inverse mapping** internally: for each output pixel, it computes where to sample from the input — so there are no holes in the result

In [ ]:
img = np.array(Image.open(IMG_DIR + "parrots_square.jpg"))

print(f"Shape: {img.shape}, dtype: {img.dtype}")
show_image(img, title="Original", scale=5)

**Rotation around the image center**: `cv2.getRotationMatrix2D` builds a $2 \times 3$ matrix that rotates around a given center point (not the origin).

In [ ]:
h, w = img.shape[:2]
center = (w // 2, h // 2)

# cv2.getRotationMatrix2D(center, angle_degrees, scale)
M_rot = cv2.getRotationMatrix2D(center, 30, 1.0)

print(f"Rotation matrix (2x3):\n{M_rot}")

rotated_img = cv2.warpAffine(img, M_rot, (w, h))
show_image(rotated_img, title="Rotated 30 deg around center", scale=5)

**Building a custom affine matrix**: we can compose our own $3 \times 3$ transforms and pass the top 2 rows to `warpAffine`.

In [ ]:
# Scale by 0.7, then rotate 20 deg, around the image center
# Strategy: translate center to origin → scale → rotate → translate back

theta = np.radians(20)
s = 0.7
cx, cy = w // 2, h // 2

T_to_origin = np.array([[1, 0, -cx], [0, 1, -cy], [0, 0, 1]], dtype=np.float64)
T_back      = np.array([[1, 0,  cx], [0, 1,  cy], [0, 0, 1]], dtype=np.float64)

R_mat = np.array([
    [np.cos(theta), -np.sin(theta), 0],
    [np.sin(theta),  np.cos(theta), 0],
    [0, 0, 1]
])

S_mat = np.array([
    [s, 0, 0],
    [0, s, 0],
    [0, 0, 1]
])

# Compose: translate to origin → scale → rotate → translate back
M_custom = T_back @ R_mat @ S_mat @ T_to_origin

print(f"Composed 3x3 matrix:\n{np.round(M_custom, 3)}")

# Pass only the top 2 rows to warpAffine
warped_custom = cv2.warpAffine(img, M_custom[:2], (w, h))
show_image(warped_custom, title="Scale(0.7) + Rotate(20 deg) around center", scale=5)

### Exercises

**Exercise 1.1:** Identify the transform type for each of the three $3 \times 3$ matrices below.

For each matrix, extract the $2 \times 2$ block $\begin{pmatrix}a & b \\ c & d\end{pmatrix}$ and check:
1. Is $a = d$ **and** $b = -c$? → could be rigid or similarity
2. If yes: is $a^2 + c^2 = 1$? → **rigid**; otherwise → **similarity**
3. If neither: → **affine**

For rigid/similarity, also compute:
- Rotation angle: $\theta = \text{atan2}(c, a)$ (use `np.degrees` to convert to degrees)
- Scale: $s = \sqrt{a^2 + c^2}$ (for rigid, $s = 1$)

*Hint:* Use `np.isclose(a, d)` for floating-point comparison.

In [ ]:
M1 = np.array([
    [0.866, -0.5,   0.5],
    [0.5,    0.866, 0.3],
    [0,      0,     1.0]
])

M2 = np.array([
    [0.693, -0.4,  0.4],
    [0.4,    0.693, 0.2],
    [0,      0,     1.0]
])

M3 = np.array([
    [1.1,  0.3,  0.2],
    [0.15, 0.9,  0.1],
    [0,    0,    1.0]
])

print(f"M1 =\n{M1}\n")
print(f"M2 =\n{M2}\n")
print(f"M3 =\n{M3}")

# YOUR CODE HERE
# For each matrix Mi, extract a, b, c, d = Mi[0,0], Mi[0,1], Mi[1,0], Mi[1,1]
# Check conditions and assign:
#   type_1, type_2, type_3 = "rigid" / "similarity" / "affine"
#   theta_1, theta_2 = angle in degrees (where applicable)
#   s_1, s_2 = scale factor (where applicable)


print(f"\nM1: {type_1}")
print(f"M2: {type_2}")
print(f"M3: {type_3}")

**Exercise 1.2:** Apply a composed transformation to the parrots image (`img`).

Build three separate $3 \times 3$ matrices in homogeneous coordinates:
1. **Scale** by factor 0.5 (both $x$ and $y$)
2. **Rotate** by 30 deg counterclockwise (around the origin)
3. **Translate** by $(100, 50)$

Compose them in the order **scale first, translate last**: $M = T \cdot R \cdot S$

Apply the resulting matrix to `img` using `cv2.warpAffine` (pass the top 2 rows of the $3 \times 3$ matrix).

*Hint:* Build each matrix as a $3 \times 3$ `np.array`, multiply with `@`, then pass `M[:2]` to `cv2.warpAffine(img, M[:2], (w, h))`.

In [ ]:
# Input
show_image(img, title="img", scale=5)

# YOUR CODE HERE
# 1. S = 3x3 scale matrix (factor 0.5)
# 2. R = 3x3 rotation matrix (30 degrees)
# 3. T = 3x3 translation matrix (tx=100, ty=50)
# 4. M = T @ R @ S
# 5. warped = cv2.warpAffine(img, M[:2], (w, h))


show_image(warped, title="Scale(0.5) -> Rotate(30 deg) -> Translate(100, 50)", scale=5)

---
## 2. Homography & Document Scanning

An **affine** transform uses a $3 \times 3$ matrix with last row $(0, 0, 1)$ — it preserves parallel lines.

A **projective** (homography) transform has a **general** last row:

$$H = \begin{pmatrix}h_{11}&h_{12}&h_{13}\\h_{21}&h_{22}&h_{23}\\h_{31}&h_{32}&1\end{pmatrix}$$

The non-zero $h_{31}, h_{32}$ create **perspective distortion** — parallel lines can converge to a vanishing point. Homography has **8 DOF**, so we need **4 point correspondences** (each gives 2 equations) to solve for $H$.

### Affine vs projective — visual comparison

Let's see the difference by warping a square grid with each transform type.

In [ ]:
# Affine warp: 3 point correspondences
src_affine = np.float32([[50, 50], [200, 50], [50, 200]])
dst_affine = np.float32([[10, 100], [200, 50], [100, 250]])

M_affine = cv2.getAffineTransform(src_affine, dst_affine)
warped_affine = cv2.warpAffine(img, M_affine, (w, h))

show_image(warped_affine, title="Affine warp (parallel lines preserved)", scale=5)

In [ ]:
# Projective (homography) warp: 4 point correspondences
src_proj = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
dst_proj = np.float32([[100, 50], [w - 100, 80], [w, h], [0, h]])

H_demo, _ = cv2.findHomography(src_proj, dst_proj)
warped_proj = cv2.warpPerspective(img, H_demo, (w, h))

show_image(warped_proj, title="Projective warp (perspective distortion)", scale=5)

In [ ]:
# Look at the homography matrix — note h31, h32 are non-zero
print(f"Homography matrix H:\n{np.round(H_demo, 6)}")
print(f"\nBottom row: [{H_demo[2, 0]:.6f}, {H_demo[2, 1]:.6f}, {H_demo[2, 2]:.6f}]")
print("  → h31, h32 != 0  →  perspective effect")

### Document scanning pipeline

The idea: given a photo of a planar object (sign, document, poster) taken at an angle, we can warp it to a **frontal view** using a homography.

**Steps:**
1. Identify 4 corner points of the object in the photo → **source points**
2. Define where those corners should map to (a rectangle) → **destination points**
3. `cv2.findHomography(src, dst)` → compute $H$
4. `cv2.warpPerspective(img, H, (w, h))` → warp the image

Let's load the signboard image and identify the corners.

In [ ]:
img_sign = np.array(Image.open(IMG_DIR + "yangbanga.jpg"))

print(f"Shape: {img_sign.shape}")

# Show with axes visible so we can read pixel coordinates
fig, ax = plt.subplots(figsize=(6, 8))
ax.imshow(img_sign)
ax.set_title("yangbanga.jpg — find the 4 corners of the signboard")
plt.tight_layout()
plt.show()

### Exercises

**Exercise 2.1:** Perform **document scanning** on the signboard in `img_sign`.

1. Define 4 **source points**: the corners of the signboard in the original image (order: top-left, top-right, bottom-right, bottom-left). Read approximate coordinates from the plot above.
2. Define 4 **destination points**: a frontal rectangle, e.g., $(0,0)$, $(300,0)$, $(300,150)$, $(0,150)$.
3. Compute the homography: `H, _ = cv2.findHomography(src_pts, dst_pts)`
4. Warp the image: `scanned = cv2.warpPerspective(img_sign, H, (output_w, output_h))`

*Hint:* The signboard corners are approximately at: top-left $\approx (115, 262)$, top-right $\approx (350, 230)$, bottom-right $\approx (362, 355)$, bottom-left $\approx (108, 385)$. Use `np.float32` for the point arrays.

In [ ]:
# Input
fig, ax = plt.subplots(figsize=(6, 8))
ax.imshow(img_sign)
ax.set_title("img_sign — identify the 4 signboard corners")
plt.tight_layout()
plt.show()

# YOUR CODE HERE
# 1. src_pts = np.float32([[x1,y1], [x2,y2], [x3,y3], [x4,y4]])  # TL, TR, BR, BL
# 2. dst_pts = np.float32([[0,0], [300,0], [300,150], [0,150]])
# 3. H, _ = cv2.findHomography(src_pts, dst_pts)
# 4. output_w, output_h = 300, 150
# 5. scanned = cv2.warpPerspective(img_sign, H, (output_w, output_h))


print(f"Homography matrix H:\n{np.round(H, 4)}")
show_image(scanned, title="Scanned signboard", scale=5)